# 01 — Data acquisition

This step pulls every game from the Chess.com API, caches the raw archives, and builds the
per-game index that the later notebooks depend on.

**Scope:** rated rapid games of standard chess only. The reasoning behind each exclusion is in
[`data/README.md`](../data/README.md).

Nothing downloaded here is committed: `data/raw/` and `data/processed/` are generated and
gitignored, and this notebook is the reproducible route to the data.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import pandas as pd

from chess_blunder import acquisition, config

pd.set_option('display.width', 120)
print(f'repo root : {REPO_ROOT}')
print(f'username  : {config.USERNAME}')
print(f'scope     : time_class={config.TIME_CLASSES} rules={config.RULES} rated_only={config.RATED_ONLY}')

## 1. List the monthly archives

The player endpoint returns one URL per calendar month, oldest first. Nothing is fetched yet —
this just tells us the range we are dealing with.

In [ ]:
archives = acquisition.list_archives()
first = '/'.join(archives[0].rsplit('/', 2)[-2:])
last = '/'.join(archives[-1].rsplit('/', 2)[-2:])
print(f'{len(archives)} monthly archives, {first} to {last}')

## 2. Download and cache the archives

Each month is written to `data/raw/YYYY-MM.json`. Months already on disk are read rather than
re-fetched, so this cell is safe to re-run and the remaining cells work offline. Requests are
paced deliberately: the API rate-limits aggressively and a burst of 46 requests will be refused.

In [ ]:
months = acquisition.download_archives(archives)
print(f'months on disk : {len(months)}')
print(f'games in total : {months.games.sum()}')
print(f'fetched now    : {int(months.fetched.sum())}')
months.head()

## 3. Describe every game before filtering

Worth seeing what is actually on the account before anything is removed. The clock column matters
most: the project's secondary question — does think time still predict blunders once position
difficulty is accounted for — is only answerable for games that carry a per-move clock.

In [ ]:
games = acquisition.load_games()
overview = acquisition.games_overview(games)
print(f'games retrieved: {len(overview)}')

by_class = overview.groupby('time_class').agg(
    games=('game_id', 'size'),
    with_clock=('has_clock', 'sum'),
)
by_class['clock_pct'] = (by_class.with_clock / by_class.games).round(4)
by_class.sort_values('games', ascending=False)

In [ ]:
rapid = overview[overview.time_class == 'rapid']
rapid.time_control.value_counts().head(8).to_frame('games')

## 4. Apply the scope filter

`build_game_index` returns the in-scope games as an index, and everything else alongside the
reason it was dropped — so the exclusion counts are visible rather than buried.

Two games that pass the scope checks turn out to be abandoned before the first move: their PGNs
contain headers but no moves. Those are caught here too, labelled `no_moves`.

In [ ]:
kept, excluded = acquisition.build_game_index(games)
print(f'in scope : {len(kept)}')
print(f'excluded : {len(excluded)}')
excluded.exclude_reason.value_counts().to_frame('games')

## 5. Structural checks

The totals are **reported, not asserted**. The current month's archive grows every time a new game
is played, so the counts are a snapshot of the retrieval date rather than a fixed property of the
data. What must hold regardless is the structure: one row per game, a timestamp on every row, no
empty games, and one clock comment per ply — the last of these is what makes the think-time feature
available for every move in the dataset rather than a subset of it.

In [ ]:
assert kept.game_id.is_unique, 'duplicate game ids in the index'
assert kept.end_time.notna().all(), 'missing end_time'
assert (kept.plies > 0).all(), 'game with no moves'
assert (kept.plies == kept.clk_comments).all(), 'clock comments do not match plies'

with_clock = int((kept.clk_comments > 0).sum())
print(f'games in scope : {len(kept)}')
print(f'date range     : {kept.end_time.min():%Y-%m-%d} to {kept.end_time.max():%Y-%m-%d}')
print(f'with clock     : {with_clock} ({with_clock / len(kept):.2%})')
print(f'plies          : min {kept.plies.min()}, median {kept.plies.median():.0f}, max {kept.plies.max()}')
print(f'total plies    : {int(kept.plies.sum())}')
print(f'rating diff    : min {kept.rating_diff.min()}, median {kept.rating_diff.median():.0f}, max {kept.rating_diff.max()}')
print('structural checks passed')

## 6. Save the index

The index is the input to labelling. It is written to `data/processed/`, which is gitignored for
the same reason as the raw archives.

In [ ]:
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
index_path = config.PROCESSED_DIR / 'games_index.csv'
kept.to_csv(index_path, index=False)
print(f'wrote {index_path.relative_to(REPO_ROOT)}')
print(f'{len(kept)} rows, {index_path.stat().st_size / 1024:.0f} KB')

## What this establishes

- Every game on the account is retrieved and cached locally, reproducibly, from a documented
  endpoint — no manual download and no committed data.
- The scope filter is applied with its exclusions counted, so the jump from the full account to
  the working dataset is visible.
- Clock coverage is measured rather than assumed. This is the number the secondary question rests
  on, and it is why rapid was chosen over the account's other time controls.
- `data/processed/games_index.csv` is the input to the next stage.

**Next:** `PYTHONPATH=src python -m chess_blunder.build_dataset --workers 8` walks each game with
Stockfish 19 and judges every move with Lichess's rules, which produces the blunder target. Settings
are fixed in `data/README.md` (100,000 nodes, `Threads` 1, `Hash` 16 MB cleared before every
position) so a re-run reproduces the same labels.